# Notebook 06 — Budget Misassignment Analysis (Section 3.5)

This notebook implements Section 3.5 of the paper: **"Assigning the uncertainty budget"**.

The key question is: what happens when the planner designs the network using an **assumed budget**
Γ_assumed ∈ {0,1,2,3,4} that may differ from the **true disruption level** Γ_true = 2?

For each (v, w, Γ_assumed) combination we:
1. Solve the robust problem (or nominal for Γ=0) to get a first-stage plan **x**.
2. Evaluate **x** under its own adversarial worst-case disruption at **Γ_true = 2**.
3. Evaluate **x** under the no-disruption scenario (cost of conservatism).
4. Evaluate **x** across N Monte Carlo scenarios sampled from Ξ(Γ_true).

The **profit loss due to misassignment** compares each plan to the correctly specified
(Γ_assumed = Γ_true = 2) benchmark, both evaluated under their respective worst cases.

**Key findings expected** (Chopra & Sodhi 2014; Lim et al. 2013):
- Over-assigning Γ is generally less costly than under-assigning.
- Profit loss from misassignment is larger when margins are tight (low v, high w).


In [ ]:
import sys, itertools, time as _time
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import matplotlib.lines as mlines

from rcflp import (
    instancemaker,
    solve_nominal,
    solve_CCG,
    evaluate_second_stage,
    evaluate_fixed_price,
    worst_case_disruption,
    no_disruption_scenario,
    sample_disruptions,
)

# ── Match the style of visualization.ipynb ────────────────────────────────────
import matplotlib as mpl
mpl.rcParams.update({
    'font.family':       'serif',
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'axes.grid':         True,
    'grid.alpha':        0.35,
    'grid.linestyle':    '--',
})

%matplotlib inline
print('Imports OK.')


In [ ]:
# ── Instance (matches paper Section 3.3 and Section 3.5) ────────────────────
In, Jn, Rn = 15, 10, 2     # customer nodes, candidate facilities, capacity levels
Hn         = 2              # disruption levels per facility (H = {0,1})
DATA_PATH  = '../dataset.xlsx'

# ── Misassignment experiment ─────────────────────────────────────────────────
GAMMA_TRUE     = 2               # true disruption budget faced in operation
GAMMA_ASSIGNED = [0, 1, 2, 3, 4] # planner's assumed budgets at design time

# ── Sensitivity grid ─────────────────────────────────────────────────────────
V_LIST = [0.75, 1.0, 1.25]    # willingness-to-pay scale
W_LIST = [10, 100, 1000]       # congestion cost

# ── Solver ───────────────────────────────────────────────────────────────────
TOL            = 0.01
TIME_LIMIT     = 6 * 3600   # total wall-clock budget per CCG run
# SUB_TIME_LIMIT matches the default time limit of worst_case_disruption (3600 s)
# so the subproblem during CCG gets the same amount of time as the post-hoc
# adversarial evaluation — eliminating the time-limit asymmetry that caused
# false convergence in earlier runs with master_time_limit=600 s.
SUB_TIME_LIMIT = 3600

# Validation threshold: flag a CCG result if the post-hoc worst-case profit
# differs from CCG's claimed lower bound by more than this fraction.
CCG_VALID_TOL  = 0.02       # 2 %

# ── Monte Carlo ──────────────────────────────────────────────────────────────
N_SAMPLES = 50
SEED      = 42

# ── Shared palette (consistent with visualization.ipynb) ─────────────────────
V_COLORS  = {0.75: '#1b7837', 1.0: '#762a83', 1.25: '#d6604d'}
V_LABELS  = {0.75: '$v=0.75$', 1.0: '$v=1.00$', 1.25: '$v=1.25$'}
W_TITLES  = {10:   '$w=10$\n(low congestion)',
             100:  '$w=100$\n(moderate)',
             1000: '$w=1000$\n(high congestion)'}

GA_COLORS = {
    0: '#d73027',   # nominal — red
    1: '#fc8d59',   # under-assign — orange-red
    2: '#4dac26',   # correct — green
    3: '#74add1',   # over-assign — light blue
    4: '#313695',   # over-assign — dark blue
}
GA_LABELS = {
    0: '$\\Gamma_a=0$ (nominal)',
    1: '$\\Gamma_a=1$ (under)',
    2: '$\\Gamma_a=2$ (correct)',
    3: '$\\Gamma_a=3$ (over)',
    4: '$\\Gamma_a=4$ (over)',
}

n_combos = len(V_LIST) * len(W_LIST) * len(GAMMA_ASSIGNED)
print(f'Instance  : I={In}, J={Jn}, R={Rn}, Hn={Hn}')
print(f'Γ_true    : {GAMMA_TRUE}')
print(f'Grid      : {n_combos} combinations')
print(f'Scenarios : {N_SAMPLES} OOS per (v,w)')
print(f'Sub TL    : {SUB_TIME_LIMIT}s  (matches post-hoc WC evaluation)')

## 1. Run the Misassignment Experiment

For each `(v, w)` pair:
1. Build instance and solve nominal.
2. For each `Γ_assumed ∈ {0,1,2,3,4}`: solve the robust (or nominal) problem → first-stage plan **x**.
3. Draw `N_SAMPLES` scenarios from Ξ(Γ_true = 2) — shared across all plans for fair comparison.
4. Evaluate each **x** under: no disruption, adversarial worst case (Γ_true = 2), and the sampled OOS scenarios.


In [ ]:
rows, oos_rows = [], []

for v, w in itertools.product(V_LIST, W_LIST):
    print(f'\n=== v={v}  w={w} ===')
    inst    = instancemaker(In, Jn, Rn, v, w, data_path=DATA_PATH)
    nom     = solve_nominal(inst)
    x_nom   = nom['x_jr']
    eps0    = no_disruption_scenario(inst, Hn)

    # Shared OOS scenarios from Ξ(Γ_true) — same for all plans at this (v,w)
    scenarios = sample_disruptions(inst, GAMMA_TRUE, Hn, N_SAMPLES, SEED)

    # First pass: collect all first-stage solutions and CCG metadata
    plans    = {}
    ccg_meta = {}   # {ga: {'profit_lb': float, 'converged': bool}}
    for ga in GAMMA_ASSIGNED:
        print(f'  Solving Γ_assumed={ga}...', end=' ', flush=True)
        t0 = _time.time()
        if ga == 0:
            plans[ga]    = x_nom
            ccg_meta[ga] = {'profit_lb': None, 'converged': True}
        else:
            res = solve_CCG(inst, ga, Hn, x_init=x_nom,
                            tol=TOL,
                            time_limit=TIME_LIMIT,
                            master_time_limit=SUB_TIME_LIMIT,
                            verbose=False)
            plans[ga]    = res['x_jr']
            # CCG minimises (negated profit), so profit_LB = -UB
            ccg_meta[ga] = {
                'profit_lb':  -res['UB'],   # claimed worst-case profit lower bound
                'converged':   res['converged'],
            }
        elapsed = _time.time() - t0
        conv_str = 'converged' if ccg_meta[ga]['converged'] else 'TIME LIMIT'
        print(f'{elapsed:.0f}s  [{conv_str}]')

    # Second pass: evaluate each plan under all three contexts
    for ga, x in plans.items():
        # No-disruption profit
        nd = evaluate_second_stage(inst, x, eps0, Hn)['profit']

        # Worst-case profit under Γ_true (each plan faces its own adversary)
        # Uses time_limit=3600 s by default — same as SUB_TIME_LIMIT, so
        # post-hoc WC and CCG's internal subproblem now operate on equal footing.
        eps_wc, _ = worst_case_disruption(inst, x, GAMMA_TRUE, Hn)
        wc        = evaluate_second_stage(inst, x, eps_wc, Hn)['profit']

        # OOS average profit
        oos_profits = [evaluate_second_stage(inst, x, eps, Hn)['profit']
                       for eps in scenarios]
        oos_arr = np.array(oos_profits)

        # ── Internal consistency check ───────────────────────────────────────
        # For ga > 0: compare post-hoc WC with CCG's claimed profit lower bound.
        # A large discrepancy means CCG's subproblem underestimated the worst case
        # (typically because it hit its time limit early) → false convergence.
        profit_lb  = ccg_meta[ga]['profit_lb']
        if profit_lb is not None:
            denom      = abs(profit_lb) + 1e-8
            diff_frac  = abs(wc - profit_lb) / denom
            ccg_valid  = (diff_frac <= CCG_VALID_TOL)
        else:
            diff_frac  = 0.0
            ccg_valid  = True   # nominal: no CCG run

        if not ccg_valid:
            print(f'    ⚠️  Γ_a={ga}: CCG gap {diff_frac*100:.1f}% '
                  f'(ccg_lb={profit_lb:,.0f}, wc={wc:,.0f}) — result suspect')

        row = dict(
            v_scale=v, w=w, gamma_assumed=ga,
            profit_nd=nd,
            profit_wc=wc,
            profit_oos_mean=float(np.mean(oos_arr)),
            profit_oos_std=float(np.std(oos_arr)),
            profit_oos_min=float(np.min(oos_arr)),
            profit_oos_pct5=float(np.percentile(oos_arr, 5)),
            profit_oos_cvar5=float(np.mean(np.sort(oos_arr)[:max(1,int(np.ceil(0.05*N_SAMPLES)))])),
            ccg_profit_lb=profit_lb,
            ccg_converged=ccg_meta[ga]['converged'],
            ccg_valid=ccg_valid,
            ccg_diff_pct=round(diff_frac * 100, 2),
        )
        rows.append(row)
        print(f'    Γ_a={ga}: nd={nd:,.0f}  wc={wc:,.0f}  avg={np.mean(oos_arr):,.0f}'
              + (f'  ⚠️ diff={diff_frac*100:.1f}%' if not ccg_valid else ''))

        for k, p in enumerate(oos_profits):
            oos_rows.append(dict(v_scale=v, w=w, gamma_assumed=ga, scenario_k=k, profit=p))

df_miss  = pd.DataFrame(rows)
df_oos   = pd.DataFrame(oos_rows)

# ── Summary of validity ──────────────────────────────────────────────────────
n_invalid = (~df_miss['ccg_valid']).sum()
if n_invalid > 0:
    print(f'\n⚠️  {n_invalid} rows flagged as suspect (CCG gap > {CCG_VALID_TOL*100:.0f}%):')
    cols = ['v_scale','w','gamma_assumed','ccg_profit_lb','profit_wc','ccg_diff_pct','ccg_converged']
    print(df_miss[~df_miss['ccg_valid']][cols].to_string(index=False))
else:
    print(f'\n✅  All CCG results consistent (internal gap ≤ {CCG_VALID_TOL*100:.0f}%).')

# Compute profit loss relative to correct design (Γ_assumed = Γ_true = 2)
ref = df_miss[df_miss['gamma_assumed'] == GAMMA_TRUE][['v_scale','w','profit_wc','profit_oos_mean']]
ref = ref.rename(columns={'profit_wc': 'ref_wc', 'profit_oos_mean': 'ref_avg'})
df_miss = df_miss.merge(ref, on=['v_scale','w'])
df_miss['loss_wc']  = df_miss['ref_wc']  - df_miss['profit_wc']
df_miss['loss_avg'] = df_miss['ref_avg'] - df_miss['profit_oos_mean']

print('\nExperiment complete.')
print(df_miss[['v_scale','w','gamma_assumed','profit_nd','profit_wc','loss_wc','ccg_valid']].to_string(index=False))

In [ ]:
EXCEL_OUT = '06_misassignment_results.xlsx'

with pd.ExcelWriter(EXCEL_OUT, engine='openpyxl') as writer:
    df_miss.to_excel(writer, sheet_name='Summary', index=False)
    df_oos.to_excel(writer,  sheet_name='OOS_Raw', index=False)

    # Pivot: profit_wc vs (v,w) for each assigned Γ
    piv_wc = df_miss.pivot_table(index='w', columns=['v_scale','gamma_assumed'],
                                  values='profit_wc')
    piv_wc.to_excel(writer, sheet_name='Pivot_WC')

    # Pivot: profit loss
    piv_loss = df_miss.pivot_table(index='w', columns=['v_scale','gamma_assumed'],
                                    values='loss_wc')
    piv_loss.to_excel(writer, sheet_name='Pivot_Loss')

    # Validity audit sheet
    audit_cols = ['v_scale','w','gamma_assumed','ccg_profit_lb','profit_wc',
                  'ccg_diff_pct','ccg_converged','ccg_valid']
    df_miss[audit_cols].to_excel(writer, sheet_name='CCG_Validity', index=False)

print(f'Saved → {EXCEL_OUT}')

# Report any suspect rows
n_inv = (~df_miss['ccg_valid']).sum()
if n_inv:
    print(f'\n⚠️  {n_inv} rows with CCG gap > {CCG_VALID_TOL*100:.0f}% saved to CCG_Validity sheet.')
    print('Consider re-running those (v,w,Γ) combinations with a longer time limit.')
else:
    print('✅  All results passed internal consistency check.')

---
## 2. Figures

### Fig 1 — Profit vs. Assigned Γ (three evaluation scenarios)

Each panel = one (v, w) combination.  
Three lines per panel: **no disruption** (solid), **worst-case Γ=2** (dashed), **OOS average Γ=2** (dotted).  
The vertical band at Γ_assumed = 2 marks the correctly specified design.


In [ ]:
fig, axes = plt.subplots(len(W_LIST), len(V_LIST), figsize=(13, 10), sharey='row')

for ri, w in enumerate(W_LIST):
    for ci, v in enumerate(V_LIST):
        ax  = axes[ri, ci]
        sub = df_miss[(df_miss['v_scale'] == v) & (df_miss['w'] == w)].sort_values('gamma_assumed')

        ga   = sub['gamma_assumed'].values
        p_nd  = sub['profit_nd'].values
        p_wc  = sub['profit_wc'].values
        p_avg = sub['profit_oos_mean'].values
        std   = sub['profit_oos_std'].values

        ax.plot(ga, p_nd,  '-o',  color='#1f78b4', lw=1.8, ms=5, label='No disruption')
        ax.plot(ga, p_wc,  '--s', color='#e31a1c', lw=1.8, ms=5, label=f'Worst-case ($\\Gamma_{{\\!true}}={GAMMA_TRUE}$)')
        ax.plot(ga, p_avg, ':^',  color='#33a02c', lw=1.8, ms=5, label=f'OOS average ($\\Gamma_{{\\!true}}={GAMMA_TRUE}$)')
        ax.fill_between(ga, p_avg - std, p_avg + std, color='#33a02c', alpha=0.12)

        ax.axvline(GAMMA_TRUE, color='gray', lw=1.2, ls='--', alpha=0.7)
        ax.axhline(0, color='black', lw=0.7, ls=':', alpha=0.5)
        ax.set_xticks(GAMMA_ASSIGNED)
        ax.set_xlabel('Assigned $\\Gamma$', fontsize=9)
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda y, _: f'${y/1e3:.0f}k'))

        if ri == 0:
            ax.set_title(V_LABELS[v], fontsize=10, color=V_COLORS[v], fontweight='bold')
        if ci == 0:
            ax.set_ylabel(f'$w={w}$\nProfit', fontsize=9)

# Shared legend
handles = [
    mlines.Line2D([], [], color='#1f78b4', lw=1.8, ls='-',  marker='o', ms=5, label='No disruption'),
    mlines.Line2D([], [], color='#e31a1c', lw=1.8, ls='--', marker='s', ms=5,
                  label=f'Worst-case ($\\Gamma_{{\\!true}}={GAMMA_TRUE}$)'),
    mlines.Line2D([], [], color='#33a02c', lw=1.8, ls=':', marker='^', ms=5,
                  label=f'OOS average ($\\Gamma_{{\\!true}}={GAMMA_TRUE}$)'),
    mlines.Line2D([], [], color='gray', lw=1.2, ls='--', label=f'$\\Gamma_{{\\!assumed}}={GAMMA_TRUE}$ (correct)'),
]
fig.legend(handles=handles, loc='lower center', ncol=4, fontsize=9,
           frameon=True, framealpha=0.92, edgecolor='0.8',
           bbox_to_anchor=(0.5, -0.03))
fig.suptitle(
    f'Impact of Budget Misassignment — Profit under Three Evaluation Scenarios\n'
    f'($\\Gamma_{{\\!true}}={GAMMA_TRUE}$; shaded band = ±1 std OOS)',
    fontsize=12, y=1.01)
fig.tight_layout()
fig.savefig('fig_misassignment_profit.pdf', bbox_inches='tight')
fig.savefig('fig_misassignment_profit.png', dpi=300, bbox_inches='tight')
plt.show()


### Fig 2 — Profit Loss Due to Misassignment

**Profit loss (WC)** = profit of the correctly-specified design (Γ=2) under its adversarial scenario  
minus profit of the misassigned design under its own adversarial scenario (both at Γ_true = 2).

Positive = the correct design does better; zero = correct assignment (Γ_assumed = 2).  
Red bars = under-assignment; blue bars = over-assignment.


In [ ]:
# ── Fig 2: 3×3 bar chart of profit loss ──────────────────────────────────────
fig, axes = plt.subplots(len(W_LIST), len(V_LIST), figsize=(13, 9), sharey='row')

BAR_COLORS = {
    0: '#d73027', 1: '#fc8d59', 2: '#4dac26', 3: '#74add1', 4: '#313695'
}
HATCHES = {0: '///', 1: '..', 2: '', 3: 'xx', 4: '\\\\\\'}

for ri, w in enumerate(W_LIST):
    for ci, v in enumerate(V_LIST):
        ax  = axes[ri, ci]
        sub = df_miss[(df_miss['v_scale'] == v) & (df_miss['w'] == w)].sort_values('gamma_assumed')

        ga   = sub['gamma_assumed'].values
        loss = sub['loss_wc'].values

        bars = ax.bar(ga, loss, width=0.6, edgecolor='black', linewidth=0.7,
                      color=[BAR_COLORS[g] for g in ga], alpha=0.80)
        for bar, g in zip(bars, ga):
            bar.set_hatch(HATCHES[g])
            bar.set_edgecolor('black')

        ax.axhline(0, color='black', lw=0.9)
        ax.set_xticks(GAMMA_ASSIGNED)
        ax.set_xticklabels([f'$\\Gamma_a={g}$' for g in GAMMA_ASSIGNED], fontsize=8)
        ax.set_xlabel('Assigned $\\Gamma$', fontsize=9)
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda y, _: f'${y/1e3:.0f}k'))

        if ri == 0:
            ax.set_title(V_LABELS[v], fontsize=10, color=V_COLORS[v], fontweight='bold')
        if ci == 0:
            ax.set_ylabel(f'$w={w}$\nProfit loss', fontsize=9)

# Shared legend
handles = [
    mpatches.Patch(facecolor=BAR_COLORS[g], hatch=HATCHES[g],
                   edgecolor='black', alpha=0.8, label=GA_LABELS[g].replace('\\\\', '\\'))
    for g in GAMMA_ASSIGNED
]
fig.legend(handles=handles, loc='lower center', ncol=5, fontsize=9,
           frameon=True, framealpha=0.92, edgecolor='0.8',
           bbox_to_anchor=(0.5, -0.04))
fig.suptitle(
    f'Worst-case Profit Loss from Budget Misassignment\n'
    f'(reference: $\\Gamma_{{\\!assumed}}={GAMMA_TRUE}$ = $\\Gamma_{{\\!true}}$; '
    f'positive = worse than correct)',
    fontsize=12, y=1.01)
fig.tight_layout()
fig.savefig('fig_misassignment_loss.pdf', bbox_inches='tight')
fig.savefig('fig_misassignment_loss.png', dpi=300, bbox_inches='tight')
plt.show()


### Fig 3 — Average Profit Loss: Under- vs Over-assignment

Averages the worst-case profit loss across all 9 (v, w) parameter combinations.  
Illustrates the key asymmetry: under-assignment (Γ < 2) is typically more costly than over-assignment (Γ > 2).


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# ── Left: average loss across all (v, w) ────────────────────────────────────
avg_loss_wc  = df_miss.groupby('gamma_assumed')['loss_wc'].mean()
avg_loss_avg = df_miss.groupby('gamma_assumed')['loss_avg'].mean()

for ax, series, title, fmt in [
    (axes[0], avg_loss_wc,  'Worst-case profit loss\n(avg across all $v,w$)',
     lambda y, _: f'${y/1e3:.1f}k'),
    (axes[1], avg_loss_avg, 'OOS average profit loss\n(avg across all $v,w$)',
     lambda y, _: f'${y/1e3:.1f}k'),
]:
    bars = ax.bar(series.index, series.values, width=0.55,
                  color=[BAR_COLORS[g] for g in series.index],
                  edgecolor='black', linewidth=0.7, alpha=0.82)
    for bar, g in zip(bars, series.index):
        bar.set_hatch(HATCHES[g])
        bar.set_edgecolor('black')

    ax.axhline(0, color='black', lw=0.9)
    ax.set_xticks(GAMMA_ASSIGNED)
    ax.set_xticklabels([f'$\\Gamma_a={g}$' for g in GAMMA_ASSIGNED])
    ax.set_ylabel('Profit loss (reference: $\\Gamma_a=2$)', fontsize=10)
    ax.set_title(title, fontsize=11)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(fmt))

    # Annotate bars
    for bar, val in zip(bars, series.values):
        y_off = max(series.values) * 0.02
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + y_off,
                f'${val/1e3:.1f}k', ha='center', va='bottom', fontsize=8.5)

# Add shading zones
for ax in axes:
    ylim = ax.get_ylim()
    ax.axvspan(-0.5, 1.5, color='#fee0d2', alpha=0.25, label='Under-assignment', zorder=0)
    ax.axvspan(1.5,  4.5, color='#deebf7', alpha=0.25, label='Over-assignment', zorder=0)
    ax.axvspan(1.5,  2.5, color='#e5f5e0', alpha=0.35, label='Correct', zorder=0)
    ax.set_ylim(ylim)

handles = [
    mpatches.Patch(color='#fee0d2', alpha=0.5, label='Under-assignment zone'),
    mpatches.Patch(color='#e5f5e0', alpha=0.5, label='Correct ($\\Gamma_a=2$)'),
    mpatches.Patch(color='#deebf7', alpha=0.5, label='Over-assignment zone'),
]
axes[1].legend(handles=handles, fontsize=8.5, frameon=True, framealpha=0.92,
               edgecolor='0.8', loc='upper right')

fig.suptitle(
    'Average Profit Loss from Budget Misassignment\n'
    '(averaged across $v \\in \\{0.75, 1.00, 1.25\\}$, $w \\in \\{10, 100, 1000\\}$)',
    fontsize=12, y=1.02)
fig.tight_layout()
fig.savefig('fig_misassignment_avg.pdf', bbox_inches='tight')
fig.savefig('fig_misassignment_avg.png', dpi=300, bbox_inches='tight')
plt.show()


### Fig 4 — OOS Violin Distributions by Assigned Γ

Profit distribution across the `N_SAMPLES` OOS scenarios for each assigned Γ,  
at the baseline (v=0.75, w=10). Shows the full risk picture: not just the mean or worst case,  
but the spread of outcomes under the true disruption environment.


In [ ]:
V_BASE, W_BASE = 0.75, 10

sub_oos = df_oos[(df_oos['v_scale'] == V_BASE) & (df_oos['w'] == W_BASE)]

fig, ax = plt.subplots(figsize=(11, 5))

x_positions = GAMMA_ASSIGNED
data_by_ga  = [sub_oos[sub_oos['gamma_assumed'] == ga]['profit'].values
               for ga in GAMMA_ASSIGNED]

vp = ax.violinplot(data_by_ga, positions=x_positions, widths=0.65,
                   showmedians=True, showextrema=True)

for patch, ga in zip(vp['bodies'], GAMMA_ASSIGNED):
    patch.set_facecolor(BAR_COLORS[ga])
    patch.set_alpha(0.70)
vp['cmedians'].set_colors('black')
vp['cmedians'].set_linewidth(2.0)
vp['cmins'].set_linewidth(1.2)
vp['cmaxes'].set_linewidth(1.2)
vp['cbars'].set_linewidth(0.8)

# Mean diamonds
for ga, data in zip(GAMMA_ASSIGNED, data_by_ga):
    ax.scatter(ga, np.mean(data), marker='D', s=50,
               color='white', edgecolor='black', zorder=5, linewidth=1.5)

ax.axvline(GAMMA_TRUE, color='gray', lw=1.4, ls='--', alpha=0.7,
           label=f'Correct assignment ($\\Gamma_{{\\!true}}={GAMMA_TRUE}$)')
ax.axhline(0, color='black', lw=0.8, ls=':', alpha=0.6)
ax.set_xticks(GAMMA_ASSIGNED)
ax.set_xticklabels([GA_LABELS[g].replace('\\\\', '\\') for g in GAMMA_ASSIGNED], fontsize=9)
ax.set_ylabel('Out-of-sample profit', fontsize=10)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda y, _: f'${y/1e3:.0f}k'))

patch_handles = [
    mpatches.Patch(facecolor=BAR_COLORS[ga], alpha=0.70,
                   label=GA_LABELS[ga].replace('\\\\', '\\'))
    for ga in GAMMA_ASSIGNED
]
patch_handles += [
    mlines.Line2D([0],[0], color='gray', lw=1.4, ls='--',
                  label=f'$\\Gamma_{{\\!assumed}}={GAMMA_TRUE}$ (correct)'),
    mlines.Line2D([0],[0], marker='D', color='w', markeredgecolor='black',
                  ms=7, lw=0, label='Mean'),
    mlines.Line2D([0],[0], color='black', lw=2, label='Median'),
]
ax.legend(handles=patch_handles, ncol=2, fontsize=8.5,
          frameon=True, framealpha=0.92, edgecolor='0.8', loc='lower right')

ax.set_title(
    f'OOS Profit Distributions by Assigned $\\Gamma$  '
    f'($v={V_BASE}$, $w={W_BASE}$, $\\Gamma_{{\\!true}}={GAMMA_TRUE}$, $N={N_SAMPLES}$)',
    fontsize=11, pad=8)
fig.tight_layout()
fig.savefig('fig_misassignment_violin.pdf', bbox_inches='tight')
fig.savefig('fig_misassignment_violin.png', dpi=300, bbox_inches='tight')
plt.show()


---
## 3. Numerical Summary

In [ ]:
# ── Table: profit loss (WC) organised as in paper Table 3 ────────────────────
piv = df_miss.pivot_table(
    index=['v_scale','w'], columns='gamma_assumed', values='loss_wc'
)
piv.index.names = ['v', 'w']
piv.columns.name = 'Γ_assumed'

fmt_dict = {col: '{:,.0f}'.format for col in piv.columns}
display(
    piv.style
    .format(fmt_dict)
    .background_gradient(cmap='RdYlGn_r', axis=None)
    .set_caption(
        f'Worst-case profit loss due to misassignment (Γ_true={GAMMA_TRUE}). '
        'Zero = correct assignment; positive = worse than correct.')
)

# ── Summary statistics ────────────────────────────────────────────────────────
print('\nAverage profit loss (WC) by assigned Γ:')
print(df_miss.groupby('gamma_assumed')['loss_wc'].agg(['mean','max']).applymap(lambda x: f'{x:,.0f}'))

under = df_miss[df_miss['gamma_assumed'] < GAMMA_TRUE]['loss_wc'].mean()
over  = df_miss[df_miss['gamma_assumed'] > GAMMA_TRUE]['loss_wc'].mean()
print(f'\nAvg loss — under-assignment: ${under:,.0f}')
print(f'Avg loss — over-assignment:  ${over:,.0f}')
print(f'Asymmetry ratio (under/over): {under/over:.2f}x' if over > 0 else '')
